In [ ]:
!pip install -q transformers datasets torch scikit-learn tqdm

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Configuration
MODEL_ID = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Updated mapping to handle plural labels from raw data
CLASS_TO_WORD = {0: "background", 1: "objective", 2: "method", 3: "result", 4: "conclusion"}
WORD_TO_CLASS = {
    'background': 0,
    'objective': 1,
    'method': 2,
    'methods': 2,
    'result': 3,
    'results': 3,
    'conclusion': 4,
    'conclusions': 4
}

print("Loading and cleaning data...")
dataset = load_dataset("armanc/pubmed-rct20k")
df_train_raw = dataset['train'].to_pandas()
df_eval_raw = dataset['validation'].to_pandas()

# Map labels and drop unknowns
df_train_raw['label'] = df_train_raw['label'].str.lower().map(WORD_TO_CLASS)
df_eval_raw['label'] = df_eval_raw['label'].str.lower().map(WORD_TO_CLASS)
df_train_raw = df_train_raw.dropna(subset=['label'])
df_eval_raw = df_eval_raw.dropna(subset=['label'])

# Ensure labels are integers
df_train_raw['label'] = df_train_raw['label'].astype(int)
df_eval_raw['label'] = df_eval_raw['label'].astype(int)

# Balanced Stratification: 1000 samples per class
def get_balanced_sample(df, samples_per_class=1000):
    balanced_df = df.groupby('label').apply(lambda x: x.sample(n=min(len(x), samples_per_class), random_state=42)).reset_index(drop=True)
    return balanced_df

print("Creating balanced 5000-sample sets...")
train_sample = get_balanced_sample(df_train_raw)
eval_sample = get_balanced_sample(df_eval_raw)

# Add abstract context
def add_abstract_context(df):
    id_col = 'abstract_id' if 'abstract_id' in df.columns else 'pmid'
    abstract_map = {aid: " ".join(group['text'].tolist()) for aid, group in df.groupby(id_col)}
    df['abstract'] = df[id_col].map(abstract_map)
    return df

train_sample = add_abstract_context(train_sample)
eval_sample = add_abstract_context(eval_sample)

print("New Distribution:")
print(eval_sample['label'].value_counts().sort_index().map(lambda x: f"{x} samples for {CLASS_TO_WORD[eval_sample[eval_sample['label']==x].index[0]] if False else 'class'}")) # simplified print
print(eval_sample.groupby('label').size())
print("Ready to re-initialize loaders and run F1 calculation.")

Loading and cleaning data...


Repo card metadata block was not found. Setting CardData to empty.


Creating balanced 5000-sample sets...


/tmp/ipykernel_3733/4107878460.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df.groupby('label').apply(lambda x: x.sample(n=min(len(x), samples_per_class), random_state=42)).reset_index(drop=True)
/tmp/ipykernel_3733/4107878460.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df.groupby('label').apply(lambda x: x.sample(n=min(len(x), samples_per_class), random_state=42

New Distribution:
label
0    1000 samples for class
1    1000 samples for class
2    1000 samples for class
3    1000 samples for class
4    1000 samples for class
Name: count, dtype: object
label
0    1000
1    1000
2    1000
3    1000
4    1000
dtype: int64
Ready to re-initialize loaders and run F1 calculation.


In [ ]:
# ==========================================
# 3. Tokenizer & Custom Dataset
# ==========================================
# Re-applying context to the new train_sample which was missing the 'abstract' column
def add_abstract_context(df):
    id_col = 'abstract_id' if 'abstract_id' in df.columns else 'pmid'
    abstract_map = {aid: " ".join(group['text'].tolist()) for aid, group in df.groupby(id_col)}
    df['abstract'] = df[id_col].map(abstract_map)
    return df

class PubMedClozeDataset(Dataset):
    def __init__(self, df):
        self.sentences = df['text'].tolist()
        self.abstracts = df['abstract'].tolist()
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        prompt = f"{self.abstracts[idx]} {tokenizer.mask_token}: {self.sentences[idx]}"
        target_token_id = VERBALIZER_TOKEN_IDS[self.labels[idx]]
        return prompt, target_token_id

def collate_fn(batch):
    prompts, target_token_ids = zip(*batch)
    inputs = tokenizer(
        list(prompts),
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    targets = torch.tensor(target_token_ids)
    return inputs, targets

def evaluate_model(model, data_loader, device):
    model.eval()
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for inputs, targets in tqdm(data_loader, desc="Evaluating"):
            inputs = {k: v.to(device) for k, v in inputs.items()}
            targets = targets.to(device)

            outputs = model(**inputs)
            logits = outputs.logits

            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)

            if mask_indices[0].size(0) == 0:
                continue

            mask_logits = logits[mask_indices[0], mask_indices[1], :]
            predicted_token_ids = torch.argmax(mask_logits, dim=-1)

            current_targets = targets[mask_indices[0]]

            correct_predictions += (predicted_token_ids == current_targets).sum().item()
            total_predictions += current_targets.size(0)

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    return accuracy

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

VERBALIZER_TOKEN_IDS = {
    label: tokenizer.convert_tokens_to_ids(word)
    for label, word in CLASS_TO_WORD.items()
}

In [ ]:
print("--- Raw Dataset Label Check ---")
print("Unique labels in train:", dataset['train'].to_pandas()['label'].unique())
print("Unique labels in validation:", dataset['validation'].to_pandas()['label'].unique())

# Let's see a few rows of what the labels actually look like
print("\nFirst 5 rows of training labels:")
print(dataset['train'].to_pandas()['label'].head())

# Re-check the mapping
print("\nMapping dictionary used:")
print(WORD_TO_CLASS)

--- Raw Dataset Label Check ---
Unique labels in train: ['objective' 'methods' 'results' 'conclusions' 'background']
Unique labels in validation: ['background' 'objective' 'methods' 'results' 'conclusions']

First 5 rows of training labels:
0    objective
1      methods
2      methods
3      methods
4      methods
Name: label, dtype: object

Mapping dictionary used:
{'background': 0, 'objective': 1, 'method': 2, 'methods': 2, 'result': 3, 'results': 3, 'conclusion': 4, 'conclusions': 4}


### BASELINE

In [ ]:
# ==========================================
# 3. Tokenizer, Custom Dataset & Training
# ==========================================
from sklearn.metrics import classification_report

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

VERBALIZER_TOKEN_IDS = {
    label: tokenizer.convert_tokens_to_ids(word)
    for label, word in CLASS_TO_WORD.items()
}

# Helper to ensure the 'abstract' column exists
def add_abstract_context(df):
    id_col = 'abstract_id' if 'abstract_id' in df.columns else 'pmid'
    abstract_map = {aid: " ".join(group['text'].tolist()) for aid, group in df.groupby(id_col)}
    df['abstract'] = df[id_col].map(abstract_map)
    return df

class PubMedClozeDataset(Dataset):
    def __init__(self, df):
        self.sentences = df['text'].tolist()
        self.abstracts = df['abstract'].tolist()
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        prompt = f"{self.abstracts[idx]} {tokenizer.mask_token}: {self.sentences[idx]}"
        target_token_id = VERBALIZER_TOKEN_IDS[self.labels[idx]]
        return prompt, target_token_id

train_dataset = PubMedClozeDataset(train_sample)
eval_dataset = PubMedClozeDataset(eval_sample)

def collate_fn(batch):
    prompts, target_token_ids = zip(*batch)
    inputs = tokenizer(
        list(prompts),
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    targets = torch.tensor(target_token_ids)
    return inputs, targets

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)
eval_loader = DataLoader(
    eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

print("Initializing PubMedBERT for Masked Language Modeling...")
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    progress_bar = tqdm(train_loader, desc="Training")

    for inputs, targets in progress_bar:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(**inputs)
            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
            if mask_indices[0].size(0) == 0: continue

            mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
            loss = loss_fn(mask_logits, targets[mask_indices[0]])

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    # Evaluation with Classification Report
    model.eval()
    all_preds, all_targets = [], []
    token_id_to_class = {v: k for k, v in VERBALIZER_TOKEN_IDS.items()}

    with torch.no_grad():
        for inputs, targets in tqdm(eval_loader, desc="Evaluating"):
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            outputs = model(**inputs)
            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
            if mask_indices[0].size(0) == 0: continue

            mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
            relevant_logits = mask_logits[:, list(VERBALIZER_TOKEN_IDS.values())]
            preds = torch.argmax(relevant_logits, dim=-1).cpu().numpy()

            # Map indices back to labels
            label_list = list(VERBALIZER_TOKEN_IDS.keys())
            all_preds.extend([label_list[p] for p in preds])

            # FIX: Ensure indices are on CPU before indexing the targets tensor
            mask_indices_cpu = mask_indices[0].cpu()
            all_targets.extend([token_id_to_class[t.item()] for t in targets[mask_indices_cpu]])

    print(f"\nEpoch {epoch+1} Metrics:")
    print(classification_report(all_targets, all_preds, target_names=[CLASS_TO_WORD[i] for i in range(5)]))

    if epoch == 1:
        torch.save(model.state_dict(), 'model_epoch_2.pt')
        print("Saved model state after Epoch 2.")

Initializing PubMedBERT for Masked Language Modeling...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from


--- Epoch 1/3 ---


Evaluating: 100%|██████████| 157/157 [01:26<00:00,  1.81it/s]



Epoch 1 Metrics:
              precision    recall  f1-score   support

  background       0.66      0.72      0.69      1000
   objective       0.79      0.58      0.67      1000
      method       0.86      0.94      0.90      1000
      result       0.85      0.92      0.89      1000
  conclusion       0.84      0.85      0.84      1000

    accuracy                           0.80      5000
   macro avg       0.80      0.80      0.80      5000
weighted avg       0.80      0.80      0.80      5000


--- Epoch 2/3 ---


Evaluating: 100%|██████████| 157/157 [01:26<00:00,  1.81it/s]



Epoch 2 Metrics:
              precision    recall  f1-score   support

  background       0.62      0.86      0.72      1000
   objective       0.85      0.52      0.64      1000
      method       0.93      0.90      0.92      1000
      result       0.92      0.86      0.89      1000
  conclusion       0.85      0.93      0.88      1000

    accuracy                           0.81      5000
   macro avg       0.83      0.81      0.81      5000
weighted avg       0.83      0.81      0.81      5000

Saved model state after Epoch 2.

--- Epoch 3/3 ---


Evaluating:  93%|█████████▎| 146/157 [01:21<00:06,  1.78it/s]

In [ ]:
import gc
import torch

# 1. Delete loaders and their references
if 'train_loader' in globals(): del train_loader
if 'eval_loader' in globals(): del eval_loader

# 2. Clear all global variables that could be holding tensors
for name in list(globals().keys()):
    if not name.startswith('__') and name not in ['gc', 'torch', 'get_ipython']:
        obj = globals()[name]
        if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
            del globals()[name]

# 3. Clear IPython history
from IPython import get_ipython
# get_ipython().history_manager.clear_output_cache() # Removed as it's not supported in Colab
get_ipython().magic('reset -f out')

# 4. Final GC and Cache clear
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("--- FINAL CLEANUP RESULT ---")
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
    if torch.cuda.memory_allocated() / 1024**2 > 1000:
        print("\nWARNING: Large amount of memory still held. Recommended: Menu -> Runtime -> Restart session")

Flushing output cache (0 entries)
--- FINAL CLEANUP RESULT ---
Allocated: 12168.62 MB
Reserved: 14706.00 MB



### CALC embed setntence

In [ ]:
!pip install -q sentence-transformers umap-learn

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP

# 1. Generate Embeddings using GPU natively
print("Generating embeddings using GPU...")
# Using BAAI/bge-small-en-v1.5 for high speed and performance
model_st = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cuda')

# Corrected variable name from df_train to df_train_raw
texts = df_train_raw['text'].tolist()

# sentence-transformers natively handles batching and progress bars
embeddings = model_st.encode(texts, batch_size=256, show_progress_bar=True)

# 2. UMAP Reduction (10 dimensions)
print("Reducing dimensions with UMAP...")
# Setting n_jobs=-1 to use available CPU threads for the reduction step
umap = UMAP(n_components=10, random_state=42, n_jobs=-1)
reduced_embeddings = umap.fit_transform(embeddings)

# 3. Representative Sampling (Closest 5000 to centroid)
centroid = reduced_embeddings.mean(axis=0).reshape(1, -1)
distances = np.linalg.norm(reduced_embeddings - centroid, axis=1)
representative_indices = np.argsort(distances)[:5000]
train_sample = df_train_raw.iloc[representative_indices].copy()

# 4. Inject Label Noise (10% total)
# Mapping for boundary/confusion noise
noise_map = {0: 1, 1: 0, 3: 4, 4: 3, 2: 3}

# 8% Boundary/Confusion Noise
boundary_noise_idx = train_sample.sample(frac=0.08, random_state=42).index
train_sample.loc[boundary_noise_idx, 'label'] = train_sample.loc[boundary_noise_idx, 'label'].map(lambda x: noise_map.get(x, (x+1)%5))

# 2% Random Noise
remaining = train_sample.drop(boundary_noise_idx)
random_noise_idx = remaining.sample(frac=0.0217, random_state=7).index
train_sample.loc[random_noise_idx, 'label'] = np.random.randint(0, 5, size=len(random_noise_idx))

print(f"New training sample size: {len(train_sample)}")
print("Label noise injected and representative sampling complete.")

Generating embeddings using GPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/691 [00:00<?, ?it/s]

Reducing dimensions with UMAP...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


New training sample size: 5000
Label noise injected and representative sampling complete.


### EVAL WITH REPR + CORRUPTION

In [ ]:
train_sample = add_abstract_context(train_sample)



train_dataset = PubMedClozeDataset(train_sample)
eval_dataset = PubMedClozeDataset(eval_sample)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)



In [ ]:


print("Initializing PubMedBERT for Masked Language Modeling...")
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    progress_bar = tqdm(train_loader, desc="Training")

    for inputs, targets in progress_bar:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(**inputs)
            logits = outputs.logits
            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)

            if mask_indices[0].size(0) == 0:
                continue

            mask_logits = logits[mask_indices[0], mask_indices[1], :]
            current_targets = targets[mask_indices[0]]
            loss = loss_fn(mask_logits, current_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_train_loss = total_loss / len(train_loader)
    val_acc = evaluate_model(model, eval_loader, DEVICE)
    print(f"Average Training Loss: {avg_train_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

    # Save the model specifically after the second epoch (index 1)
    if epoch == 1:
        torch.save(model.state_dict(), 'model_epoch_2.pt')
        print("Saved model state after Epoch 2.")

Initializing PubMedBERT for Masked Language Modeling...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from


--- Epoch 1/3 ---


Evaluating: 100%|██████████| 157/157 [01:39<00:00,  1.58it/s]


Average Training Loss: 0.9110 | Validation Accuracy: 0.7632

--- Epoch 2/3 ---


Evaluating: 100%|██████████| 157/157 [01:39<00:00,  1.58it/s]


Average Training Loss: 0.5568 | Validation Accuracy: 0.7686
Saved model state after Epoch 2.

--- Epoch 3/3 ---


Evaluating: 100%|██████████| 157/157 [01:38<00:00,  1.59it/s]

Average Training Loss: 0.4651 | Validation Accuracy: 0.7574


### REPR EVAL

#### EVAL LOADER

In [ ]:
# Re-creating the evaluation loader for the final stratified evaluation
# Using the eval_sample (balanced 5000-sample set) previously defined

eval_dataset = PubMedClozeDataset(eval_sample)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(f"Eval loader ready with {len(eval_sample)} samples.")

Eval loader ready with 5000 samples.


In [ ]:
import os
from sklearn.metrics import classification_report
from transformers import AutoConfig

# 1. Per-Label Representative Sampling
print("Calculating per-label centroids and selecting closest samples...")
total_n = 5000
label_counts = df_train_raw['label'].value_counts(normalize=True)
representative_indices = []

for label, fraction in label_counts.items():
    n_samples = int(round(fraction * total_n))
    # Get indices for this label
    label_mask = (df_train_raw['label'] == label).values
    label_indices = np.where(label_mask)[0]

    # Calculate centroid for this label's embeddings
    label_embeddings = reduced_embeddings[label_indices]
    centroid = label_embeddings.mean(axis=0).reshape(1, -1)

    # Find closest n samples to the label centroid
    dists = np.linalg.norm(label_embeddings - centroid, axis=1)
    closest_relative_indices = np.argsort(dists)[:n_samples]
    representative_indices.extend(label_indices[closest_relative_indices])

train_sample_repr = df_train_raw.iloc[representative_indices].copy().reset_index(drop=True)
train_sample_repr = add_abstract_context(train_sample_repr)

# 2. Re-initialize Fresh Model and Train
print(f"Initializing fresh model for training on {len(train_sample_repr)} representative samples...")
config = AutoConfig.from_pretrained(MODEL_ID)
config.tie_word_embeddings = False
model_repr = AutoModelForMaskedLM.from_pretrained(MODEL_ID, config=config).to(DEVICE)

repr_loader = DataLoader(
    PubMedClozeDataset(train_sample_repr),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

optimizer = torch.optim.AdamW(model_repr.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

# Training Loop
for epoch in range(EPOCHS):
    model_repr.train()
    total_loss = 0
    for inputs, targets in tqdm(repr_loader, desc=f"Training Epoch {epoch+1}"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model_repr(**inputs)
            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
            if mask_indices[0].size(0) == 0: continue
            mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
            loss = loss_fn(mask_logits, targets[mask_indices[0]])
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(repr_loader):.4f}")

# 3. Final Evaluation on 5k Stratified Eval Set
print("\n--- FINAL EVALUATION ON 5K STRATIFIED EVAL SET ---")
final_acc = evaluate_model(model_repr, eval_loader, DEVICE)
print(f"Validation Accuracy: {final_acc:.4f}")

# 4. Detailed Classification Report
all_preds, all_targets = [], []
token_id_to_class = {v: k for k, v in VERBALIZER_TOKEN_IDS.items()}
model_repr.eval()
with torch.no_grad():
    for inputs, targets in eval_loader:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        outputs = model_repr(**inputs)
        mask_indices = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)
        if mask_indices[0].size(0) == 0: continue
        mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
        pred_token_ids = torch.argmax(mask_logits, dim=-1).cpu().numpy()
        target_token_ids = targets[mask_indices[0]].cpu().numpy()
        for p, t in zip(pred_token_ids, target_token_ids):
            if p in token_id_to_class and t in token_id_to_class:
                all_preds.append(token_id_to_class[p])
                all_targets.append(token_id_to_class[t])

print(classification_report(all_targets, all_preds, target_names=[CLASS_TO_WORD[i] for i in range(5)]))

Calculating per-label centroids and selecting closest samples...
Initializing fresh model for training on 5000 representative samples...


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Training Epoch 1: 100%|██████████| 157/157 [01:16<00:00,  2.06it/s]


Epoch 1 Avg Loss: 0.5936


...

Training Epoch 3: 100%|██████████| 157/157 [01:14<00:00,  2.11it/s]


Epoch 3 Avg Loss: 0.1718

--- FINAL EVALUATION ON 5K STRATIFIED EVAL SET ---


Evaluating: 100%|██████████| 157/157 [01:27<00:00,  1.80it/s]

Validation Accuracy: 0.7490


              precision    recall  f1-score   support

  background       0.54      0.77      0.64      1000
   objective       0.76      0.49      0.60      1000
      method       0.81      0.93      0.87      1000
      result       0.93      0.78      0.85      1000
  conclusion       0.80      0.77      0.79      1000

    accuracy                           0.75      5000
   macro avg       0.77      0.75      0.75      5000
weighted avg       0.77      0.75      0.75      5000



### 15k CORR

In [ ]:
# 1. Sample 15k Random Points
print("Sampling 15,000 random samples...")
train_sample_15k = df_train_raw.sample(n=15000, random_state=42).copy()

# 2. Inject 10% Label Noise (similar distribution to representative exp)
# 8% Boundary/Confusion Noise + 2% Random Noise
noise_map = {0: 1, 1: 0, 3: 4, 4: 3, 2: 3}
boundary_noise_idx = train_sample_15k.sample(frac=0.08, random_state=42).index
train_sample_15k.loc[boundary_noise_idx, 'label'] = train_sample_15k.loc[boundary_noise_idx, 'label'].map(lambda x: noise_map.get(x, (x+1)%5))

remaining = train_sample_15k.drop(boundary_noise_idx)
random_noise_idx = remaining.sample(frac=0.0217, random_state=7).index
train_sample_15k.loc[random_noise_idx, 'label'] = np.random.randint(0, 5, size=len(random_noise_idx))

# 3. Prepare Data Context and Loaders
train_sample_15k = add_abstract_context(train_sample_15k)
train_loader_15k = DataLoader(
    PubMedClozeDataset(train_sample_15k),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

Sampling 15,000 random samples...


In [ ]:
# import gc
# import torch
# import sys

# def deep_nuclear_cleanup():
#     print("--- DEEP GPU MEMORY PURGE ---")

#     # 1. Clear known large objects from global namespace
#     vars_to_kill = [
#         'model', 'model_repr', 'model_15k', 'optimizer', 'scaler',
#         'train_loader', 'eval_loader', 'train_loader_15k', 'repr_loader',
#         'all_probs', 'pred_probs', 'outputs', 'mask_logits', 'logits', 'relevant_logits',
#         'embeddings', 'reduced_embeddings', 'embeddings_15k'
#     ]

#     for var in vars_to_kill:
#         if var in globals():
#             print(f"Deleting {var}...")
#             del globals()[var]

#     # 2. Force garbage collection
#     gc.collect()
#     gc.collect()

#     # 3. Hard reset of PyTorch CUDA cache (re-importing locally to avoid UnboundLocalError)
#     import torch
#     if torch.cuda.is_available():
#         torch.cuda.synchronize()
#         torch.cuda.empty_cache()
#         torch.cuda.reset_peak_memory_stats()

#         allocated = torch.cuda.memory_allocated() / 1024**2
#         reserved = torch.cuda.memory_reserved() / 1024**2

#         print(f"\nFinal Memory Status:")
#         print(f"Allocated: {allocated:.2f} MB")
#         print(f"Reserved:  {reserved:.2f} MB")

#         if allocated > 500:
#             print("\n[!] WARNING: GPU memory is still fragmented or leaked.")
#             print("Please go to: Runtime -> Restart Session")
#         else:
#             print("\nSUCCESS: GPU memory is clear. You can proceed with the 15k training.")

# deep_nuclear_cleanup()

--- DEEP GPU MEMORY PURGE ---
Deleting eval_loader...
Deleting train_loader_15k...
Deleting embeddings...
Deleting reduced_embeddings...

Final Memory Status:
Allocated: 136.39 MB
Reserved:  318.00 MB

SUCCESS: GPU memory is clear. You can proceed with the 15k training.


In [ ]:
import gc
import torch
from transformers import AutoConfig
from sklearn.metrics import classification_report

# 1. Clear memory to avoid OOM
if 'model_15k' in globals(): del model_15k
if 'model_repr' in globals(): del model_repr
gc.collect()
torch.cuda.empty_cache()

# 2. Re-initialize loaders with smaller batch size for 15k set
BATCH_SIZE_REDUCED = 16
train_loader_15k = DataLoader(
    PubMedClozeDataset(train_sample_15k),
    batch_size=BATCH_SIZE_REDUCED,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# 3. Re-initialize and Train
print("Initializing fresh model for 15k random experiment (Reduced Batch Size)...")
config = AutoConfig.from_pretrained(MODEL_ID)
config.tie_word_embeddings = False
model_15k = AutoModelForMaskedLM.from_pretrained(MODEL_ID, config=config).to(DEVICE)
optimizer = torch.optim.AdamW(model_15k.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

for epoch in range(EPOCHS):
    model_15k.train()
    total_loss = 0
    progress_bar = tqdm(train_loader_15k, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for inputs, targets in progress_bar:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model_15k(**inputs)
            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
            if mask_indices[0].size(0) == 0: continue
            mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
            loss = loss_fn(mask_logits, targets[mask_indices[0]])

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    val_acc = evaluate_model(model_15k, eval_loader, DEVICE)
    print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(train_loader_15k):.4f} | Val Acc: {val_acc:.4f}")

# 4. Final Evaluation
print("\n--- Final Metrics for 15k Random Noisy Dataset ---")
all_preds, all_targets = [], []
token_id_to_class = {v: k for k, v in VERBALIZER_TOKEN_IDS.items()}
model_15k.eval()
with torch.no_grad():
    for inputs, targets in tqdm(eval_loader, desc="Generating Report"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        outputs = model_15k(**inputs)
        mask_indices = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)
        if mask_indices[0].size(0) == 0: continue
        mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
        pred_token_ids = torch.argmax(mask_logits, dim=-1).cpu().numpy()
        target_token_ids = targets[mask_indices[0]].cpu().numpy()
        for p, t in zip(pred_token_ids, target_token_ids):
            if p in token_id_to_class and t in token_id_to_class:
                all_preds.append(token_id_to_class[p])
                all_targets.append(token_id_to_class[t])

print("\nClassification Report:")
print(classification_report(all_targets, all_preds, target_names=[CLASS_TO_WORD[i] for i in range(5)]))

Initializing fresh model for 15k random experiment (Reduced Batch Size)...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Evaluating: 100%|██████████| 157/157 [01:32<00:00,  1.69it/s]


Epoch 1 Avg Loss: 0.7169 | Val Acc: 0.7970


Evaluating: 100%|██████████| 157/157 [01:35<00:00,  1.64it/s]


Epoch 2 Avg Loss: 0.5642 | Val Acc: 0.7840


Evaluating: 100%|██████████| 157/157 [01:36<00:00,  1.63it/s]


Epoch 3 Avg Loss: 0.4657 | Val Acc: 0.7806

--- Final Metrics for 15k Random Noisy Dataset ---


Generating Report: 100%|██████████| 157/157 [01:36<00:00,  1.62it/s]



Classification Report:
              precision    recall  f1-score   support

  background       0.59      0.73      0.65      1000
   objective       0.71      0.56      0.63      1000
      method       0.88      0.95      0.91      1000
      result       0.85      0.92      0.88      1000
  conclusion       0.91      0.76      0.83      1000

    accuracy                           0.78      5000
   macro avg       0.79      0.78      0.78      5000
weighted avg       0.79      0.78      0.78      5000



### 15k CORR2

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM, AutoConfig
from sklearn.metrics import classification_report
from tqdm import tqdm

# 1. Dataset Setup (15k Random Samples)
print("Sampling 15,000 random training points...")
train_sample_consistent = df_train_raw.sample(n=15000, random_state=42).copy()

# 2. Consistent Corruption (5% swap for Background<->Objective and Results<->Conclusion)
def apply_consistent_corruption(df, swap_pairs, rate=0.05):
    df['original_label'] = df['label']
    df['is_corrupted'] = False

    for c1, c2 in swap_pairs:
        # Identify indices for class 1 and class 2
        idx1 = df[df['label'] == c1].index
        idx2 = df[df['label'] == c2].index

        # Sample 5% from each to swap
        swap_idx1 = np.random.choice(idx1, size=int(len(idx1) * rate), replace=False)
        swap_idx2 = np.random.choice(idx2, size=int(len(idx2) * rate), replace=False)

        # Perform the swap
        df.loc[swap_idx1, 'label'] = c2
        df.loc[swap_idx2, 'label'] = c1
        df.loc[list(swap_idx1) + list(swap_idx2), 'is_corrupted'] = True

    return df

# Swap pairs: (0: Background, 1: Objective), (3: Result, 4: Conclusion)
swap_pairs = [(0, 1), (3, 4)]
train_sample_consistent = apply_consistent_corruption(train_sample_consistent, swap_pairs)
train_sample_consistent = add_abstract_context(train_sample_consistent)

# 3. Loaders
consistent_loader = DataLoader(
    PubMedClozeDataset(train_sample_consistent),
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

# 4. Training Loop
print("Training model on consistently corrupted 15k dataset...")
config = AutoConfig.from_pretrained(MODEL_ID)
config.tie_word_embeddings = False
model_consistent = AutoModelForMaskedLM.from_pretrained(MODEL_ID, config=config).to(DEVICE)
optimizer = torch.optim.AdamW(model_consistent.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler('cuda')

for epoch in range(2): # 2 Epochs for speed
    model_consistent.train()
    for inputs, targets in tqdm(consistent_loader, desc=f"Epoch {epoch+1}"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model_consistent(**inputs)
            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
            if mask_indices[0].size(0) == 0: continue
            loss = torch.nn.CrossEntropyLoss()(outputs.logits[mask_indices], targets[mask_indices[0]])
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

# 5. Evaluation
print("\n--- Final Metrics for Consistent 15k Corruption ---")
val_acc = evaluate_model(model_consistent, eval_loader, DEVICE)
print(f"Validation Accuracy: {val_acc:.4f}")

Sampling 15,000 random training points...
Training model on consistently corrupted 15k dataset...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 2: 100%|██████████| 938/938 [03:20<00:00,  4.68it/s]



--- Final Metrics for Consistent 15k Corruption ---


Evaluating: 100%|██████████| 157/157 [01:35<00:00,  1.64it/s]

Validation Accuracy: 0.8116


In [ ]:
print("--- Classification Report: 15k Consistent Corruption ---")
all_preds, all_targets = [], []
token_id_to_class = {v: k for k, v in VERBALIZER_TOKEN_IDS.items()}
model_consistent.eval()

with torch.no_grad():
    for inputs, targets in tqdm(eval_loader, desc="Generating Report"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        outputs = model_consistent(**inputs)
        mask_indices = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)
        if mask_indices[0].size(0) == 0: continue
        mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
        pred_token_ids = torch.argmax(mask_logits, dim=-1).cpu().numpy()
        target_token_ids = targets[mask_indices[0]].cpu().numpy()
        for p, t in zip(pred_token_ids, target_token_ids):
            if p in token_id_to_class and t in token_id_to_class:
                all_preds.append(token_id_to_class[p])
                all_targets.append(token_id_to_class[t])

print("\nClassification Report:")
print(classification_report(all_targets, all_preds, target_names=[CLASS_TO_WORD[i] for i in range(5)]))

--- Classification Report: 15k Consistent Corruption ---


Generating Report: 100%|██████████| 157/157 [01:22<00:00,  1.90it/s]


Classification Report:
              precision    recall  f1-score   support

  background       0.65      0.76      0.70      1000
   objective       0.78      0.61      0.68      1000
      method       0.90      0.94      0.92      1000
      result       0.86      0.94      0.90      1000
  conclusion       0.89      0.81      0.85      1000

    accuracy                           0.81      5000
   macro avg       0.82      0.81      0.81      5000
weighted avg       0.82      0.81      0.81      5000



### 20k initial

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoConfig
from sklearn.metrics import classification_report
from tqdm import tqdm

# 1. Stratified Sampling (20k total, 4k per class)
print("Sampling 20,000 stratified samples...")
def get_stratified_sample(df, total_samples=20000):
    n_classes = df['label'].nunique()
    samples_per_class = total_samples // n_classes
    return df.groupby('label').apply(lambda x: x.sample(n=min(len(x), samples_per_class), random_state=42)).reset_index(drop=True)

train_sample_20k = get_stratified_sample(df_train_raw)
train_sample_20k = add_abstract_context(train_sample_20k)

# 2. Dataset & Loaders
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
VERBALIZER_TOKEN_IDS = {label: tokenizer.convert_tokens_to_ids(word) for label, word in CLASS_TO_WORD.items()}

class PubMedDataset(Dataset):
    def __init__(self, df):
        self.sentences = df['text'].tolist()
        self.abstracts = df['abstract'].tolist()
        self.labels = df['label'].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return f"{self.abstracts[idx]} {tokenizer.mask_token}: {self.sentences[idx]}", VERBALIZER_TOKEN_IDS[self.labels[idx]]

def collate_fn(batch):
    texts, labels = zip(*batch)
    inputs = tokenizer(list(texts), padding=True, truncation=True, max_length=512, return_tensors="pt")
    return inputs, torch.tensor(labels)

train_loader_20k = DataLoader(PubMedDataset(train_sample_20k), batch_size=16, shuffle=True, collate_fn=collate_fn)
eval_loader_standard = DataLoader(PubMedDataset(eval_sample), batch_size=16, shuffle=False, collate_fn=collate_fn)

# 3. Model Initialization
print("Initializing model...")
config = AutoConfig.from_pretrained(MODEL_ID)
config.tie_word_embeddings = False
model_20k = AutoModelForMaskedLM.from_pretrained(MODEL_ID, config=config).to(DEVICE)
optimizer = torch.optim.AdamW(model_20k.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler('cuda')

# 4. Training (2 Epochs)
for epoch in range(2):
    model_20k.train()
    total_loss = 0
    pbar = tqdm(train_loader_20k, desc=f"Epoch {epoch+1}/2")
    for inputs, targets in pbar:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        targets = targets.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model_20k(**inputs)
            mask_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
            if mask_indices[0].size(0) == 0: continue
            logits = outputs.logits[mask_indices[0], mask_indices[1], :]
            loss = torch.nn.CrossEntropyLoss()(logits, targets[mask_indices[0]])
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

# 5. Evaluation & Report
print("\n--- Final Evaluation ---")
model_20k.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in tqdm(eval_loader_standard, desc="Evaluating"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        outputs = model_20k(**inputs)
        mask_indices = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)
        if mask_indices[0].size(0) == 0: continue
        logits = outputs.logits[mask_indices[0], mask_indices[1], :]
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        truth = targets[mask_indices[0]].cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(truth)

# Map token IDs back to class names for report
id_to_name = {v: CLASS_TO_WORD[k] for k, v in VERBALIZER_TOKEN_IDS.items()}
print(classification_report(all_targets, all_preds,
                            target_names=[id_to_name[VERBALIZER_TOKEN_IDS[i]] for i in range(5)]))

Sampling 20,000 stratified samples...


/tmp/ipykernel_2388/180173514.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('label').apply(lambda x: x.sample(n=min(len(x), samples_per_class), random_state=42)).reset_index(drop=True)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 2/2: 100%|██████████| 1250/1250 [04:38<00:00,  4.49it/s, loss=0.3917]



--- Final Evaluation ---


Evaluating:   0%|          | 0/313 [00:00<?, ?it/s]


RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [ ]:
# Final Evaluation Report for the 20k Stratified Experiment
print("Generating Classification Report for the 20k model...")
model_20k.eval()
all_preds, all_targets = [], []

# Map target token IDs to 0-4 indices
token_to_idx = {token_id: idx for idx, token_id in VERBALIZER_TOKEN_IDS.items()}

with torch.no_grad():
    for inputs, targets in tqdm(eval_loader_standard, desc="Final Eval"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        outputs = model_20k(**inputs)
        mask_indices = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)

        if mask_indices[0].size(0) == 0: continue

        logits = outputs.logits[mask_indices[0], mask_indices[1], :]

        # Subset logits to only our 5 verbalizer tokens
        relevant_logits = logits[:, list(VERBALIZER_TOKEN_IDS.values())]
        preds = torch.argmax(relevant_logits, dim=-1).cpu().numpy()

        # Move targets to CPU for consistent indexing and conversion
        targets_cpu = targets.cpu()
        mask_indices_cpu = mask_indices[0].cpu()

        truth = [token_to_idx[targets_cpu[i].item()] for i in mask_indices_cpu]

        all_preds.extend(preds)
        all_targets.extend(truth)

print("\n20k Stratified Experiment - Classification Report:")
print(classification_report(all_targets, all_preds, target_names=[CLASS_TO_WORD[i] for i in range(5)]))

Generating Classification Report for the 20k model...


Final Eval: 100%|██████████| 313/313 [01:20<00:00,  3.88it/s]


20k Stratified Experiment - Classification Report:
              precision    recall  f1-score   support

  background       0.68      0.78      0.72      1000
   objective       0.78      0.65      0.71      1000
      method       0.92      0.94      0.93      1000
      result       0.93      0.88      0.90      1000
  conclusion       0.88      0.94      0.91      1000

    accuracy                           0.84      5000
   macro avg       0.84      0.84      0.83      5000
weighted avg       0.84      0.84      0.83      5000



### Cleanlab Audit on Consistent Corruption
We will now use Cleanlab to see if it can identify the 5% systematic swaps between (Background/Objective) and (Result/Conclusion).

In [ ]:
import numpy as np
from cleanlab import Datalab

# 1. Generate probabilities for the consistently corrupted set
model_consistent.eval()
all_probs = []

consistent_eval_loader = DataLoader(
    PubMedClozeDataset(train_sample_consistent),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print("Generating probabilities for Cleanlab...")
with torch.no_grad():
    for inputs, _ in tqdm(consistent_eval_loader):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        outputs = model_consistent(**inputs)
        mask_indices = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)
        if mask_indices[0].size(0) == 0: continue

        mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
        # Map model vocabulary back to our 5 labels
        relevant_logits = mask_logits[:, list(VERBALIZER_TOKEN_IDS.values())]
        probs = torch.softmax(relevant_logits, dim=-1).cpu().numpy()
        all_probs.append(probs)

pred_probs_consistent = np.vstack(all_probs)

# 2. Run Cleanlab Datalab
lab = Datalab(data=train_sample_consistent, label_name="label")
lab.find_issues(pred_probs=pred_probs_consistent, issue_types={"label": {}})
issue_summary = lab.get_issues("label")

# 3. Report Detection Accuracy
print("\n--- Cleanlab Detection Performance (Consistent Noise) ---")
print(classification_report(
    train_sample_consistent['is_corrupted'],
    issue_summary['is_label_issue'],
    target_names=['Clean', 'Corrupted']
))

Generating probabilities for Cleanlab...


100%|██████████| 469/469 [04:17<00:00,  1.82it/s]


Finding label issues ...

Audit complete. 718 issues found in the dataset.

--- Cleanlab Detection Performance (Consistent Noise) ---
              precision    recall  f1-score   support

       Clean       0.99      0.97      0.98     14502
   Corrupted       0.45      0.65      0.53       498

    accuracy                           0.96     15000
   macro avg       0.72      0.81      0.76     15000
weighted avg       0.97      0.96      0.97     15000



### PSEUDO + CLEANLAB

In [ ]:
!pip install -q cleanlab

import numpy as np
from cleanlab import Datalab
from sklearn.metrics import classification_report

# 1. Prepare the Corrupted dataset probs using model_repr
print("Generating probabilities for corrupted 15k dataset using the representative-trained model...")
model_repr.eval()
all_probs = []

# Note: train_sample_15k already contains 10% noise from earlier cell
corrupted_dataset = PubMedClozeDataset(train_sample_15k)
corrupted_loader = DataLoader(
    corrupted_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

with torch.no_grad():
    for inputs, _ in tqdm(corrupted_loader, desc="Predicting Probs"):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        outputs = model_repr(**inputs)
        mask_indices = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)

        if mask_indices[0].size(0) == 0: continue

        # Extract logits for the [MASK] and convert to probabilities
        mask_logits = outputs.logits[mask_indices[0], mask_indices[1], :]
        # We only care about the 5 class tokens defined in VERBALIZER_TOKEN_IDS
        relevant_logits = mask_logits[:, list(VERBALIZER_TOKEN_IDS.values())]
        probs = torch.softmax(relevant_logits, dim=-1).cpu().numpy()
        all_probs.append(probs)

pred_probs = np.vstack(all_probs)
labels = train_sample_15k['label'].values

# 2. Use Cleanlab to find issues
print("Running Cleanlab Datalab...")
lab = Datalab(data=train_sample_15k, label_name="label")
lab.find_issues(pred_probs=pred_probs, issue_types={"label": {}})

issue_summary = lab.get_issues("label")

# 3. Validation: Compare against ground truth corruption
# We identify corrupted rows by comparing 'label' to the original uncorrupted labels if we had them
# Since we didn't save the 'true' labels before noise injection in that specific df,
# let's calculate based on the indices we recorded earlier (boundary_noise_idx and random_noise_idx)

train_sample_15k['is_actually_corrupted'] = False
# Note: These indices are from the original df_train_raw
train_sample_15k.loc[train_sample_15k.index.isin(boundary_noise_idx), 'is_actually_corrupted'] = True
train_sample_15k.loc[train_sample_15k.index.isin(random_noise_idx), 'is_actually_corrupted'] = True

true_corruption = train_sample_15k['is_actually_corrupted'].values
detected_corruption = issue_summary['is_label_issue'].values

print("\n--- Cleanlab Corruption Detection Report ---")
print(classification_report(true_corruption, detected_corruption, target_names=['Clean', 'Corrupted']))

Generating probabilities for corrupted 15k dataset using the representative-trained model...


NameError: name 'model_repr' is not defined

### UMAP APPROACH FOR CORRUPTION

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

# 1. Get embeddings for the 15k samples
# Since reduced_embeddings was for the whole 176k training set, we index it using the 15k sample indices
indices_15k = train_sample_15k.index.values
embeddings_15k = reduced_embeddings[indices_15k]

# 2. Calculate Per-Class Centroids in UMAP space (based on the noisy labels)
centroids = {}
for label in range(5):
    mask = (train_sample_15k['label'] == label).values
    if mask.any():
        centroids[label] = embeddings_15k[mask].mean(axis=0)

# 3. Calculate Distance of each sample to its assigned class centroid
distances_to_centroid = []
for i, row in enumerate(train_sample_15k.itertuples()):
    label = row.label
    embedding = embeddings_15k[i]
    centroid = centroids[label]
    dist = np.linalg.norm(embedding - centroid)
    distances_to_centroid.append(dist)

train_sample_15k['umap_dist'] = distances_to_centroid

# 4. Flag the top 10% (matching our noise injection rate) as 'detected corruption'
# This is a 'purely geometric' detection heuristic
threshold = train_sample_15k['umap_dist'].quantile(0.90)
detected_umap_corruption = train_sample_15k['umap_dist'] >= threshold

# 5. Evaluate against Ground Truth
print("--- UMAP Geometric Corruption Detection Report ---")
print(classification_report(
    train_sample_15k['is_actually_corrupted'],
    detected_umap_corruption,
    target_names=['Clean', 'Corrupted']
))

# Analysis of intersection
umap_detected_set = set(train_sample_15k[detected_umap_corruption].index)
cleanlab_detected_set = set(train_sample_15k[issue_summary['is_label_issue'].values].index)
intersection = umap_detected_set.intersection(cleanlab_detected_set)

print(f"UMAP detected: {len(umap_detected_set)}")
print(f"Cleanlab detected: {len(cleanlab_detected_set)}")
print(f"Commonly identified samples: {len(intersection)}")

--- UMAP Geometric Corruption Detection Report ---
              precision    recall  f1-score   support

       Clean       0.90      0.90      0.90     13501
   Corrupted       0.13      0.13      0.13      1499

    accuracy                           0.83     15000
   macro avg       0.52      0.52      0.52     15000
weighted avg       0.83      0.83      0.83     15000

UMAP detected: 1500
Cleanlab detected: 3735
Commonly identified samples: 438
